In [ ]:
!pip install torch torchvision torch-geometric


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 61.9 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)


100%|██████████| 9.91M/9.91M [00:00<00:00, 134MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 36.9MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 105MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.87MB/s]


In [ ]:
def image_to_graph(img, label):

    img = img.view(28,28)

    xs = []

    for i in range(28):
        for j in range(28):

            intensity = img[i,j]

            x_pos = i / 27.0
            y_pos = j / 27.0

            dx = img[i,j] - img[i,j-1] if j > 0 else 0
            dy = img[i,j] - img[i-1,j] if i > 0 else 0

            xs.append([intensity, x_pos, y_pos, dx, dy])

    x = torch.tensor(xs, dtype=torch.float)



    edges = []
    for i in range(28):
        for j in range(28):

            node = i*28 + j

            for di in [-1,0,1]:
                for dj in [-1,0,1]:

                    if di == 0 and dj == 0:
                        continue

                    ni = i + di
                    nj = j + dj

                    if 0 <= ni < 28 and 0 <= nj < 28:

                        neighbor = ni*28 + nj
                        edges.append([node, neighbor])

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return Data(x=x, edge_index=edge_index, y=torch.tensor(label))


In [ ]:
train_graphs = [image_to_graph(train_dataset[i][0], train_dataset[i][1])
                for i in range(5000)]
test_graphs = [image_to_graph(test_dataset[i][0], test_dataset[i][1])
               for i in range(1000)]

In [ ]:
train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=32)


In [ ]:
class GNN(torch.nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = GATConv(5,64,heads=4)
        self.conv2 = GATConv(256,128,heads=4)
        self.conv3 = GATConv(512,256,heads=1)

        self.bn1 = torch.nn.BatchNorm1d(256)
        self.bn2 = torch.nn.BatchNorm1d(512)
        self.bn3 = torch.nn.BatchNorm1d(256)

        self.fc = torch.nn.Linear(512,10)


    def forward(self,data):

        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x,edge_index)
        x = self.bn1(x)
        x = F.relu(x)

        x = self.conv2(x,edge_index)
        x = self.bn2(x)
        x = F.relu(x)

        x = self.conv3(x,edge_index)
        x = self.bn3(x)
        x = F.relu(x)

        x_mean = global_mean_pool(x,batch)
        x_max = global_max_pool(x,batch)

        x = torch.cat([x_mean,x_max],dim=1)

        x = F.dropout(x,p=0.5,training=self.training)

        return self.fc(x)


In [ ]:
model = GNN().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.5
)


In [ ]:
def train():

    model.train()

    total_loss = 0

    for data in train_loader:

        data = data.to(device)

        optimizer.zero_grad()

        out = model(data)

        loss = F.cross_entropy(out,data.y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)


In [ ]:
@torch.no_grad()
def test(loader):

    model.eval()

    correct = 0

    for data in loader:

        data = data.to(device)

        out = model(data)

        pred = out.argmax(dim=1)

        correct += int((pred == data.y).sum())

    return correct / len(loader.dataset)


In [ ]:
epochs = 30

for epoch in range(1,epochs+1):

    loss = train()

    acc = test(test_loader)

    scheduler.step()

    print(f"Epoch {epoch:02d} | Loss {loss:.4f} | Test Acc {acc:.4f}")


Epoch 01 | Loss 1.4667 | Test Acc 0.9130
Epoch 02 | Loss 0.4895 | Test Acc 0.9460
Epoch 03 | Loss 0.3342 | Test Acc 0.9560
Epoch 04 | Loss 0.2763 | Test Acc 0.9570


KeyboardInterrupt: 